In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import soundfile as sf

from IPython.display import Audio, display

In [2]:
BASE_DIR = Path.cwd().parent

CLEAN_DIR = BASE_DIR / "data/processed/clean_speech"

NOISY_DIR = BASE_DIR / "data/processed/noisy_speech/white_noise"

REPORT_DIR = BASE_DIR / "results/metrics"

NOISY_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

TARGET_SR = 16000

SNR_LEVELS = [0, 5, 10]

print("Clean speech:", CLEAN_DIR)
print("Noisy speech:", NOISY_DIR)

Clean speech: d:\Kuliah\Pengolahan Sinyal Digital\PROJECT\data\processed\clean_speech
Noisy speech: d:\Kuliah\Pengolahan Sinyal Digital\PROJECT\data\processed\noisy_speech\white_noise


In [3]:
clean_files = sorted(CLEAN_DIR.glob("*.wav"))

print("Jumlah clean speech:", len(clean_files))

for file in clean_files[:5]:
    print(file.name)

Jumlah clean speech: 20
common_voice_common_voice_id_19192818.wav
common_voice_common_voice_id_20338960.wav
common_voice_common_voice_id_20387848.wav
common_voice_common_voice_id_20899278.wav
common_voice_common_voice_id_20952818.wav


In [4]:
def add_white_noise(clean_audio, target_snr_db, seed=7):

    # Membuat white noise acak
    rng = np.random.default_rng(seed)

    noise = rng.normal(
        0,
        1,
        size=len(clean_audio)
    )

    # Menghitung daya sinyal bersih
    signal_power = np.mean(clean_audio ** 2)

    # Menghitung daya noise awal
    noise_power = np.mean(noise ** 2)

    if signal_power <= 0 or noise_power <= 0:
        raise ValueError("Sinyal memiliki daya nol.")

    # Menghitung daya noise sesuai target SNR
    target_noise_power = (
        signal_power / (10 ** (target_snr_db / 10))
    )

    # Mengatur amplitudo noise
    noise = noise * np.sqrt(
        target_noise_power / noise_power
    )

    # Mencampurkan clean speech dengan noise
    noisy_audio = clean_audio + noise

    # Menghitung SNR aktual
    actual_snr = 10 * np.log10(
        np.mean(clean_audio ** 2) /
        np.mean(noise ** 2)
    )

    return noisy_audio, noise, actual_snr

In [5]:
results = []

for file_index, clean_path in enumerate(clean_files):

    # Membaca clean speech
    clean_audio, sr = sf.read(clean_path)

    if clean_audio.ndim != 1:
        raise ValueError(
            f"Audio bukan mono: {clean_path.name}"
        )

    if sr != TARGET_SR:
        raise ValueError(
            f"Sample rate tidak sesuai: {clean_path.name}"
        )

    for snr in SNR_LEVELS:

        noisy_audio, noise, actual_snr = add_white_noise(
            clean_audio,
            target_snr_db=snr,
            seed=7 + file_index
        )

        # Nama file output
        output_name = (
            f"{clean_path.stem}_SNR{snr}dB.wav"
        )

        output_path = NOISY_DIR / output_name

        # Simpan sebagai WAV float agar tidak clipping
        sf.write(
            output_path,
            noisy_audio,
            sr,
            subtype="FLOAT"
        )

        results.append({
            "clean_filename": clean_path.name,
            "noisy_filename": output_name,
            "noise_type": "white_noise",
            "target_snr_db": snr,
            "actual_snr_db": round(float(actual_snr), 4),
            "duration_seconds": round(
                len(noisy_audio) / sr, 3
            ),
            "sample_rate": sr,
            "output_path": str(output_path)
        })

print("Proses noise generation selesai!")

Proses noise generation selesai!


In [6]:
noise_df = pd.DataFrame(results)

report_path = REPORT_DIR / "noise_generation_report.csv"

noise_df.to_csv(report_path, index=False)

print("Jumlah noisy speech:", len(noise_df))
print("Laporan:", report_path)

display(noise_df.head(10))

Jumlah noisy speech: 60
Laporan: d:\Kuliah\Pengolahan Sinyal Digital\PROJECT\results\metrics\noise_generation_report.csv


,clean_filename,noisy_filename,noise_type,target_snr_db,actual_snr_db,duration_seconds,sample_rate,output_path
0,common_voice_common_voice_id_19192818.wav,common_voice_common_voice_id_19192818_SNR0dB.wav,white_noise,0,0.0,5.112,16000,d:\Kuliah\Pengolahan Sinyal Digital\PROJECT\da...
1,common_voice_common_voice_id_19192818.wav,common_voice_common_voice_id_19192818_SNR5dB.wav,white_noise,5,5.0,5.112,16000,d:\Kuliah\Pengolahan Sinyal Digital\PROJECT\da...
2,common_voice_common_voice_id_19192818.wav,common_voice_common_voice_id_19192818_SNR10dB.wav,white_noise,10,10.0,5.112,16000,d:\Kuliah\Pengolahan Sinyal Digital\PROJECT\da...
3,common_voice_common_voice_id_20338960.wav,common_voice_common_voice_id_20338960_SNR0dB.wav,white_noise,0,0.0,6.648,16000,d:\Kuliah\Pengolahan Sinyal Digital\PROJECT\da...
4,common_voice_common_voice_id_20338960.wav,common_voice_common_voice_id_20338960_SNR5dB.wav,white_noise,5,5.0,6.648,16000,d:\Kuliah\Pengolahan Sinyal Digital\PROJECT\da...
5,common_voice_common_voice_id_20338960.wav,common_voice_common_voice_id_20338960_SNR10dB.wav,white_noise,10,10.0,6.648,16000,d:\Kuliah\Pengolahan Sinyal Digital\PROJECT\da...
6,common_voice_common_voice_id_20387848.wav,common_voice_common_voice_id_20387848_SNR0dB.wav,white_noise,0,0.0,5.256,16000,d:\Kuliah\Pengolahan Sinyal Digital\PROJECT\da...
7,common_voice_common_voice_id_20387848.wav,common_voice_common_voice_id_20387848_SNR5dB.wav,white_noise,5,5.0,5.256,16000,d:\Kuliah\Pengolahan Sinyal Digital\PROJECT\da...
8,common_voice_common_voice_id_20387848.wav,common_voice_common_voice_id_20387848_SNR10dB.wav,white_noise,10,10.0,5.256,16000,d:\Kuliah\Pengolahan Sinyal Digital\PROJECT\da...
9,common_voice_common_voice_id_20899278.wav,common_voice_common_voice_id_20899278_SNR0dB.wav,white_noise,0,0.0,5.136,16000,d:\Kuliah\Pengolahan Sinyal Digital\PROJECT\da...


In [ ]:
# Ambil satu contoh clean speech
sample_clean = clean_files[0]

clean_audio, sr = sf.read(sample_clean)

# Cari noisy speech dengan SNR 10 dB
sample_noisy_path = NOISY_DIR / (
    f"{sample_clean.stem}_SNR10dB.wav"
)

noisy_audio, _ = sf.read(sample_noisy_path)

print("CLEAN SPEECH")
display(Audio(clean_audio, rate=sr))

print("NOISY SPEECH - SNR 10 dB")
display(Audio(noisy_audio, rate=sr))

CLEAN SPEECH


NOISY SPEECH - SNR 10 dB
